In [16]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [17]:
yerevan_historical = pd.read_csv('../data/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../data/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [18]:
yerevan_historical.drop(columns=['sold_date', 'days_in'], inplace=True)

In [19]:
yerevan_actual['backup_status'] = 'not sold'

In [20]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [21]:
df[df['id']==10895]

,backup_status,id,price,rooms,square_meters,address,source,furniture,renovation,price_per_meter,...,links,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood
33,sold,10895,100000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",real-estate.am,1.0,NaN,1851.851852,...,https://www.real-estate.am/en/2-bedrooms/apart...,2022-08-28 19:49:22,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513121,40.166179,NaN
18279,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",real-estate.am,1.0,NaN,2314.814815,...,https://www.real-estate.am/en/2-bedrooms/apart...,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513110,40.166191,Kentron


In [22]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [23]:
df = df.dropna(subset=['district']).reset_index(drop=True)

## Price change table

In [24]:
df1 = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [25]:
df1[df1['id']==10895]

,backup_status,id,price,rooms,square_meters,address,source,furniture,renovation,price_per_meter,...,links,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood
33,sold,10895,100000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",real-estate.am,1.0,NaN,1851.851852,...,https://www.real-estate.am/en/2-bedrooms/apart...,2022-08-28 19:49:22,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513121,40.166179,NaN
18279,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",real-estate.am,1.0,NaN,2314.814815,...,https://www.real-estate.am/en/2-bedrooms/apart...,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513110,40.166191,Kentron


In [26]:
df1.loc[df1.duplicated(subset=['id'], keep=False), 'backup_status'] = 'price change'
df1 = df1.drop_duplicates(subset=['id'], keep='last')

In [27]:
df1 = df1.dropna(subset=['district']).reset_index(drop=True)

### classifying neighborhoods

In [29]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)

In [31]:
df = sale_historical.head(61046).reset_index(drop=True)

In [34]:
import requests

api_key = "***REMOVED-GOOGLE-API-KEY***"

# addresses = []

In [45]:
while True:
    try:
        for index, row in df.iterrows():
            latlng = f"{row['x']},{row['y']}"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            # print(address)
            df.drop(index=df.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...


In [44]:
df

,backup_status,id,price,rooms,square_meters,address,source,furniture,renovation,price_per_meter,...,links,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood
78,sold,38524,120000.0,4.0,216.0,"Yerevan, Norq Marash, Norq 1 St",NaN,NaN,NaN,NaN,...,https://www.real-estate.am/en/house-for-sale/N...,NaN,44.547185,40.188708,Yeritasardakan,3844,Norq_Marash,44.547184,40.188710,NaN
79,sold,38677,430000.0,5.0,140.0,"Yerevan, Center, Antarayin St",NaN,NaN,NaN,NaN,...,https://www.real-estate.am/en/house-for-sale/A...,NaN,44.514693,40.192083,Yeritasardakan,2194,Kentron,44.514701,40.192080,NaN
80,sold,39629,130000.0,1.0,42.0,"Yerevan, Center, Tpagrichner St",real-estate.am,1.0,NaN,3095.238095,...,https://www.real-estate.am/en/studio/apartment...,2022-08-28 19:49:20,44.519715,40.178014,Republic Square,662,Kentron,44.519708,40.178015,NaN
81,sold,39868,102000.0,4.0,108.0,"Yerevan, Davtashen, Davtashen 2 district",NaN,NaN,NaN,NaN,...,https://www.real-estate.am/en/3-bedrooms/apart...,NaN,44.495065,40.225053,Barekamutyun,4008,Davtashen,44.495069,40.225052,NaN
82,sold,40671,137000.0,3.0,105.0,"Yerevan, Arabkir, Adonts St",NaN,NaN,NaN,NaN,...,https://www.real-estate.am/en/2-bedrooms/apart...,NaN,44.522958,40.212396,Baghramyan,3969,Arabkir,44.522951,40.212393,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61041,sold,19119793,100000.0,5.0,NaN,"Մուշ թաղամաս, Երևան",list.am,0.0,Euro Renovation,617.283951,...,http://list.am/en/item/19119793,2023-03-14 12:19:54,44.468161,40.181239,Barekamutyun,4232,Malatia_Sebastia,44.468157,40.181244,NaN
61042,sold,19120368,88000.0,6.0,NaN,"Արագածի փողոց, Երևան",list.am,0.0,Partial Renovation,275.000000,...,http://list.am/en/item/19120368,2023-03-14 12:19:47,44.484258,40.159855,Garegin Nzhdeh Square,1366,Shengavit,44.484265,40.159856,NaN
61043,sold,19120399,88000.0,6.0,NaN,"Արագածի փողոց, Երևան",list.am,0.0,Partial Renovation,273.291925,...,http://list.am/en/item/19120399,2023-03-14 12:19:45,44.484258,40.159855,Garegin Nzhdeh Square,1366,Shengavit,44.484261,40.159848,NaN
61044,sold,19121390,90000.0,4.0,NaN,"Նորագավիթ 13-րդ փողոց, Երևան",list.am,0.0,No Renovation,204.545455,...,http://list.am/en/item/19121390,2023-03-14 11:50:11,44.466885,40.113095,Charbakh,5101,Shengavit,44.466877,40.113086,NaN


In [39]:
len(addresses)

78

In [ ]:
sale_historical['neighborhood'] = addresses

In [ ]:
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [ ]:
df = pd.concat([sale_historical, sale_actual], axis=0)

In [ ]:
df2 = df.head(61046).reset_index(drop=True)

In [ ]:
import requests

api_key = "***REMOVED-GOOGLE-API-KEY***"

# addresses = []

In [ ]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            # print(address)
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

In [15]:
df_clean = df.dropna().reset_index(drop=True)

In [35]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)
df_clean.to_csv(r'../data/yerevan_combined_clean/sale_clean_combined.csv', index=False)
df1.to_csv(r'../data/yerevan_combined_clean/sale_combined_price_change.csv', index=False)

In [ ]:
df[df['backup_status']=='sold'].reset_index(drop=True).to_csv(r'../data/yerevan_combined_clean/historical_sold.csv', index=False)
df[df['backup_status']=='not sold'].reset_index(drop=True).to_csv(r'../data/yerevan_combined_clean/actual_not_sold.csv', index=False)
